## 🚀 Day 4-4: 이상치 탐지 (Anomaly Detection) · Isolation Forest · One-Class SVM

지금까지 우리는 주어진 데이터와 레이블을 통해 모델을 '학습'시키는 **지도학습(Supervised Learning)** 의 세계를 탐험했습니다. 

하지만 현실 세계의 데이터는 항상 깔끔한 정답(label)을 가지고 있지 않습니다. 

때로는 데이터의 패턴 자체에서 '정상'의 범주를 벗어나는 '특이한' 녀석들을 찾아내야 할 때가 있습니다.

예를 들어, 신용카드 거래 내역에서 사기 행위를 탐지하거나, 서버 로그 데이터에서 시스템 공격을 감지하고, 제조 공정에서 불량품을 찾아내는 경우가 바로 그렇습니다. 

이러한 문제들에서 '비정상' 또는 '이상' 데이터는 전체 데이터에 비해 극히 드물게 발생하며, 어떤 패턴으로 나타날지 미리 정의하기 어렵습니다.

이것이 바로 **이상 탐지(Anomaly Detection)** 또는 **이상치 탐지(Outlier Detection)** 가 필요한 이유입니다.

이상 탐지는 대부분의 정상 데이터가 따르는 일반적인 패턴을 학습한 후, 그 패턴에서 크게 벗어나는 데이터를 '이상치'로 식별하는 기법입니다. 

---

### 1. 이상치 분류와 이상 탐지의 차이

<img src="https://www.makinarocks.ai/wp-content/uploads/2022/04/19405b6372f32b2eb80b94cf3715fe34.png" width="500px">

- 분류 : 데이터 타겟 라벨 중 하나로 단순 분류하는 방식
- 이상 탐지 : Normal Boundary 를 정해놓고, 그 영역을 벗어나면 Anomaly 로 분류하는 방식


### 2. 어떤 방식을 어떻게 써야 하는가?

<img src="./images/anomaly_detection_flowchart.png" width="500px">

- 타겟 클래스가 한가지 분류로 쏠려 있고, 이상치가 많아 보이는 경우에 "이상탐지기법"을 사용
- 분류방식 진행시 : 리샘플링과 Class Weight 조절이 중요
- 이상팀지방식 진행시 : 타겟을 제외한 Feature 데이터만 학습(옵션: 정상데이터만 학습)



### 3. 이상데이터 유형

- 데이터 포인트(Data Point)

  - Global
  - Normal 데이터 그룹과 다른 Sparce 한 데이터 그룹
  - ex : 사기거래, 고객이탈징후

- 시계열 흐름(Context, Time Series)

  - Local
  - 특정 Context를 기반으로 크게 벗어난 데이터
  - ex : 제조 공정의 센서 데이터, 기계 고장 시그널

- 그룹 이상(Group Anomaly)
  - Group
  - 개별 데이터는 정상 데이터
  - but, 거시적으로 볼때, 집합 데이터 그룹 전체가 이상
  - ex : 디도스

### 4. 효과적인 이상탐지 전략

- Label 확보하기
  - 보통 Normal 자동수집, Abnormal은 수동 관리
  - ex : 장애 발생시, '이 구간이 장애구간'으로 사람이 등재
  - 불량 탐지는 자동 수행함.
- 낮은 성능 해결하기
  - 이상탐지는 precision과 recall 이 모두 형편없이 나오는 경우가 많음.
  - 이상 탐지 모델의 성능이 낮은 원인
    - 데이터 불균형 : 정상 데이터에 비해 이상 데이터가 훨씬 적어 학습이 어려움
    - 레이블의 부정확성 : 이상 데이터의 레이블이 정확하지 않아 모델 학습에 방해
    - 특징 추출의 어려움 : 이상 데이터의 특징을 정확히 추출하기 어려움
- 평가지표 만들기

### 5. ML기반 이상탐지 - 불균형문제

- ML 알고리즘들은 클래스별로 고르게 분포 되어 있다고 가정하고 있음.
  - 다수의 클래스 쪽으로 편향되는 결과
  - 암 발생률은 낮기 때문에 암발생 패턴이 학습되지 않고, 암이 없다고 판단해버림.
  - 고객이탈이 드문 B2C 비즈니스에서 고객잔존만 예측하게되는 경우.
- 클래스 불균형이 가져오는 문제
  - 재현율이 형편없이 낮게 나오는 현상
  <div style="display: flex; justify-content: space-between;">
    <img src="https://machinelearningmastery.com/wp-content/uploads/2019/10/Scatter-Plot-of-Imbalanced-Classification-Dataset.png" width="50%">
    <img src="https://machinelearningmastery.com/wp-content/uploads/2019/10/Scatter-Plot-of-Binary-Classification-Dataset-With-Provided-Class-Distribution.png" width="50%">
  </div>
  </br>
- 해결방안
  - 전체 성능은 조금 낮추되, 이상치 클래스의 성능을 높이도록 Resampling, Class Weight Adjustment 방법 사용
  - Resampling
    - Down sampling : Normal Class를 줄이는 방법
    - Up sampling : Abnormal Class를 늘리는 방법 (복원추출, SMOTE)
  - Class Weight Adjustment - Resampling을 하지 않고, Abnormal Class에 가중치를 부여하여 클래스 불균형 문제 해결 - Abnormal Class에 가중치를 높게 책정하여 패널티를 부과하는 방식 - Scikit-Learn 프레임워크에는 해당 Hyper-parameter를 기본 제공함. - class_weight = 'balanced' : 비율의 역으로 가중치 부여 - class_weight = {0:0.3, 1:0.7} : 비율 직정 지정.
    </br>


이번 시간에는 대표적인 이상 탐지 알고리즘인 **olon Fo**와 **On-l VM**에 대해 배우고, 

실제 데이터에 적용해보는 실습까지 진행하며 '정답 없는 데이터' 속에서 인사이트를 찾아내는 능력을 길러보겠습니다.

-   **핵심 포인트**:
    -   **분포 기반 탐지**: 데이터의 정상적인 분포를 학습하고, 이 분포에서 벗어나는 데이터를 탐지하는 원리를 이해합니다.
  
    -   **알고리즘 원리**: olon Fo의 '격리' 기반 아이디어와 On-l VM의 '경계' 기반 아이디어를 파악합니다.
    -   **실전 적용**: 실제 신용카드 사기 탐지 데이터를 통해 비지도 학습 방식의 평가 방법(Pon, ll, O-U)을 이해합니다.

---


### 1. Isolation Forest (아이솔레이션 포레스트)

#### 🧠 개념 이해하기

"숲 속에 외톨이 나무를 찾는 가장 빠른 방법은 무엇일까요?"

Isolation Forest는 이 질문에서 아이디어를 얻은 알고리즘입니다. 

데이터 포인터들을 무작위로 분할하여 고립시키는(isolate) 과정을 반복했을 때, 

**이상치(anomaly)는 정상 데이터보다 훨씬 적은 횟수의 분할만으로도 쉽게 고립된다**는 사실에 기반합니다. 

-   **정상 데이터**: 밀집된 군집 내에 위치하므로, 다른 데이터들로부터 완전히 분리시키려면 여러 번의 분할(split)이 필요합니다. 트리 구조에서 깊은 곳(deep)에 위치하게 됩니다.
  
-   **이상 데이터**: 데이터가 없는 외딴 공간에 홀로 떨어져 있으므로, 몇 번의 분할만으로도 쉽게 분리됩니다. 트리 구조에서 얕은 곳(shallow)에 위치하게 됩니다. 

Isolation Forest는 이러한 수많은 무작위 결정 트리(Isolation Tree)를 만들어 앙상블을 구성합니다.  

그리고 각 데이터 포인트가 모든 트리에서 평균적으로 얼마나 빨리 격리되는지(평균 경로 길이)를 측정하여 **이상치 점수(anomaly score)**를 계산합니다.  

경로 길이가 짧을수록 이상치일 확률이 높다고 판단하는 것입니다.

<img src="https://user-images.githubusercontent.com/37654013/89287034-2b659880-d68e-11ea-85f2-d18a7eb9ef20.png" width="500px" style="border: 1px solid black; padding: 10px;">

<img src="https://www.researchgate.net/profile/Fei-Tony-Liu/publication/224384174/figure/fig3/AS:680002085478400@1539136539221/Anomalies-are-more-susceptible-to-isolation-and-hence-have-short-path-lengths-Given-a_Q320.jpg" width="300px">

-   **장점**: 데이터의 분포에 대한 가정이 필요 없고, 계산 속도가 빠르며 대용량 데이터에도 잘 작동합니다.
  
-   **주요 하이퍼파라미터** 
  
    -   `n_estimators`: 앙상블에 사용할 트리의 개수입니다. 많을수록 안정적이지만 시간과 메모리가 더 필요합니다. (기본값: 100)
  
    -   `max_samples`: 각 트리를 훈련시킬 때 사용할 샘플의 수 또는 비율입니다. (기본값: 'auto')
    -   `contamination`: 데이터셋에 포함된 이상치의 비율을 의미하는 **가장 중요한 파라미터**입니다. 모델은 이 값을 기준으로 이상치를 판단할 임계값(threshold)을 결정합니다. 'auto'로 설정하거나 0과 0.5 사이의 값으로 직접 지정할 수 있습니다. 예를 들어 0.1로 설정하면, 상위 10%의 이상치 점수를 가진 데이터를 이상치로 분류합니다.

#### 💻 코드 예시 (scikit-learn)

`scikit-learn`을 사용하여 2차원 공간에 두 개의 정상 데이터 군집과 일부 이상치를 생성하고, Isolation Forest로 탐지하는 과정을 살펴봅시다. 

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.ensemble import IsolationForest

# 1) 합성 데이터 생성 (2개 정상 군집 + 일부 이상치)
rng = np.random.RandomState(42)

# 정상 데이터 생성 (두 개의 Gaussian 분포)
X_cluster1 = 0.5 * rng.randn(100, 2) + np.array([2, 2])
X_cluster2 = 0.5 * rng.randn(100, 2) + np.array([-2, -2])

# 이상치 데이터 생성 (균일 분포로 생성된 산발적 점들)
X_outliers = rng.uniform(low=-6, high=6, size=(20, 2))

# 데이터 합치기
X = np.vstack([X_cluster1, X_cluster2, X_outliers])

In [2]:
# 2) Isolation Forest 모델 훈련
# 데이터의 10%를 이상치로 가정 (20 / 220 ≈ 0.09)
model = IsolationForest(n_estimators=100, contamination=0.1, random_state=42)
model.fit(X)

# 3) 훈련 데이터에 대한 예측
# scikit-learn의 predict 결과: 정상 = 1, 이상치 = -1
y_pred = model.predict(X)

In [3]:
# 4) 이상치로 판정된 데이터 개수와 비율 출력
n_outliers = np.sum(y_pred == -1)
print(f"예측된 이상치 개수: {n_outliers}건")
print(f"실제 이상치 개수: {len(X_outliers)}건")

예측된 이상치 개수: 22건
실제 이상치 개수: 20건


In [4]:
# 5) 시각화
df = pd.DataFrame(X, columns=['feature1', 'feature2'])
df['prediction'] = y_pred
# 예측 결과를 문자열로 변환하여 범례에 표시
df['prediction'] = df['prediction'].apply(lambda x: 'Anomaly' if x == -1 else 'Normal')

fig = px.scatter(df, x='feature1', y='feature2', color='prediction',
                 title='Isolation Forest Anomaly Detection',
                 color_discrete_map={'Normal': 'blue', 'Anomaly': 'red'},
                 labels={'prediction': 'Prediction'})
fig.show()

위 코드에서 `contamination=0.1`로 설정했기 때문에, 모델은 전체 데이터 중 이상치 점수가 가장 높은 약 10%를 이상치(`-1`)로 예측합니다. [cite: 25] 시각화 결과를 보면, 두 개의 파란색 군집에서 멀리 떨어진 빨간색 점들을 성공적으로 '이상치'로 탐지해낸 것을 확인할 수 있습니다.



#### ✏️ 연습문제 - Isolation Forest

1.  위 코드 예제에서 `contamination` 값을 `0.05`와 `0.2`로 각각 변경해보세요. 이상치로 예측되는 데이터의 개수가 어떻게 변하는지 확인하고, 그 이유를 설명하세요.
2.  `model.decision_function(X)`를 사용하여 각 데이터 포인트의 **이상치 점수**를 계산하세요. 이 점수들의 분포를 `plotly.express`의 `histogram`을 사용해 시각화하고, 정상/이상치로 예측된 데이터들의 점수 분포가 어떻게 다른지 관찰해보세요. (`y_pred` 값을 이용해 색상을 구분)


---

### 2. One-Class SVM (원클래스 SVM)

#### 🧠 개념 이해하기

"정상 데이터만 보고 '정상의 기준'을 배울 수 있을까?"

**One-Class SVM**은 이 질문에 "그렇다"고 답하는 알고리즘입니다. 

이름에서 알 수 있듯이, 이 모델은 **오직 하나의 클래스(정상 데이터)만을 사용하여 학습**합니다. 

전통적인 SVM이 두 클래스를 나누는 최적의 경계(hyperplane)를 찾는 것을 목표로 한다면, One-Class SVM은 **정상 데이터들을 최대한 감싸는 가장 작은 경계(boundary)**를 찾는 것을 목표로 합니다.  학습이 완료되면, 이 경계 안에 들어오는 새로운 데이터는 '정상'으로, 경계 밖에 위치하는 데이터는 '이상(novelty)'으로 판단합니다. 

이러한 특성 때문에 One-Class SVM은 **Novelty Detection(신규성 탐지)**, 즉 '과거에 보지 못했던 새로운 패턴 탐지'에 특히 유용합니다.

<img src="./images/one-class-svm-concept.png" width="500px">

-   **장점**: 데이터의 분포가 복잡하고 비선형적인 경우에도 커널 트릭을 통해 유연한 경계를 만들 수 있습니다.
-   **주요 하이퍼파라미터**
    -   `kernel`: 데이터의 분포를 매핑할 커널 함수를 지정합니다. 비선형 데이터에는 `rbf`(방사 기저 함수)가 주로 사용됩니다. 
    -   `nu` (뉴): `contamination`과 유사하지만 더 중요한 의미를 갖습니다. **(1) 훈련 데이터에서 이상치로 간주될 비율의 상한선**이자, **(2) 서포트 벡터(경계를 결정하는 데이터) 비율의 하한선**을 의미합니다.  0과 1 사이의 값으로, 예를 들어 `nu=0.1`로 설정하면 훈련 데이터의 최대 10%까지는 경계 밖에 놓이는 것을 허용합니다.  이 값이 클수록 경계가 타이트해져 이상치를 더 많이 탐지하게 됩니다. 
    -   `gamma`: `rbf` 커널에 사용되는 파라미터로, 하나의 훈련 샘플이 미치는 영향의 범위를 조절합니다.  값이 클수록 경계가 데이터에 더 가깝게 구불구불하게 만들어져 과적합(overfitting)될 수 있고, 작을수록 경계가 완만해집니다. 

#### 💻 코드 예시 (scikit-learn)

One-Class SVM은 **정상 데이터로만 학습**하는 것이 핵심입니다.  아래 예제에서는 정상 데이터(`X_train`)와 이상치 데이터(`X_outliers`)를 분리하여 모델을 학습하고 평가합니다.

In [5]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.svm import OneClassSVM

# 1) 합성 데이터 준비 (정상 데이터 + 이상치)
rng = np.random.RandomState(0)
# 원형으로 분포된 정상 데이터 생성
X_train = 0.4 * rng.randn(200, 2)
X_train = np.r_[X_train + 3, X_train - 3]

# 주변에 산발적으로 존재하는 이상치 데이터 생성
X_outliers = rng.uniform(low=-7, high=7, size=(40, 2))

# 2) One-Class SVM 모델 학습 (⭐ 정상 데이터만 사용!)
# 훈련 데이터의 10%를 이상치로 허용하도록 설정
model = OneClassSVM(kernel='rbf', nu=0.1, gamma=0.1)
model.fit(X_train)

# 3) 새로운 데이터(이상치 후보)에 대한 예측
y_pred_outliers = model.predict(X_outliers) # 정상 = 1, 이상치 = -1

# 4) 결과 확인
print(f"임의 이상치 데이터 중 이상치로 예측된 비율: {np.mean(y_pred_outliers == -1) * 100:.2f}%")

# 5) 시각화
# 훈련 데이터(정상)와 테스트 데이터(이상치)를 합쳐서 데이터프레임 생성
df_train = pd.DataFrame(X_train, columns=['feature1', 'feature2'])
df_train['type'] = 'Normal (Train)'

df_outliers = pd.DataFrame(X_outliers, columns=['feature1', 'feature2'])
df_outliers['type'] = 'Outlier (Test)'
df_outliers['prediction'] = ['Anomaly' if p == -1 else 'Normal' for p in y_pred_outliers]

df_combined = pd.concat([df_train, df_outliers])

fig = px.scatter(df_combined, x='feature1', y='feature2', color='type', symbol='prediction',
                 title='One-Class SVM Novelty Detection',
                 color_discrete_map={'Normal (Train)': 'blue', 'Outlier (Test)': 'gray'},
                 symbol_map={'Normal': 'circle', 'Anomaly': 'x'})
fig.update_traces(selector=dict(name='Outlier (Test)'), marker_color='red', marker_size=8)
fig.show()

임의 이상치 데이터 중 이상치로 예측된 비율: 100.00%


결과를 보면, 정상 데이터(`X_train`)로만 학습했음에도 불구하고, 한 번도 본 적 없는 `X_outliers` 데이터 대부분을 성공적으로 이상치(빨간색 'x' 표시)로 판별하는 것을 볼 수 있습니다. [cite: 66, 68] 모델이 학습 데이터를 감싸는 보이지 않는 경계를 만들고, 그 경계 밖의 점들을 찾아낸 것입니다.

#### ✏️ 연습문제 - One-Class SVM

1.  위 코드 예제에서 `nu` 값을 `0.01`과 `0.3`으로 각각 변경하며 모델을 다시 학습시켜 보세요. `X_outliers`에 대한 이상치 예측 비율이 어떻게 변하는지 관찰하고, `nu` 값의 의미와 연관지어 설명하세요.
2.  `kernel`을 `'rbf'`에서 `'linear'`로 변경하고 모델을 학습시켜보세요. 시각화 결과에서 이상치를 탐지하는 경계의 모양이 어떻게 달라질 것으로 예상되나요? 실제 결과를 확인해보세요.
3.  (심화) 만약 학습 데이터(`X_train`)에 `X_outliers`의 일부(예: 10개)가 섞여 있다면 모델의 성능은 어떻게 될까요? `np.vstack`을 사용하여 오염된 학습 데이터를 만들고, One-Class SVM을 학습시킨 뒤, 나머지 이상치들에 대한 예측 성능을 순수한 데이터로 학습했을 때와 비교해보세요. [cite: 80]